In [3]:
import pandas as pd
import os

# Load the rationales data
data_path = "../../crest/data/rationales/e_movies_test_sparsemap_50p.tsv"
df = pd.read_csv(data_path, sep='\t')


In [7]:
import numpy as np
import ast

def process_text_and_rationales(text, z_values):
    """
    Convert tokenized text to normal form and aggregate rationales at word level.
    
    Args:
        text (str): Tokenized text with ▁ markers
        z_values (list): Binary rationale values for each token
    
    Returns:
        tuple: (normal_text, word_rationales, rationale_ranges)
    """
    # Split text into tokens
    tokens = text.split()
    
    # Parse z_values if it's a string representation of a list
    if isinstance(z_values, str):
        z_values = ast.literal_eval(z_values)
    
    words = []
    word_rationales = []
    current_word = ""
    current_word_has_rationale = False
    
    for i, token in enumerate(tokens):
        z_val = z_values[i] if i < len(z_values) else 0.0
        
        if token.startswith('▁'):
            # Start of a new word
            if current_word:
                # Save the previous word
                words.append(current_word)
                word_rationales.append(1 if current_word_has_rationale else 0)
            
            # Start new word (remove ▁ marker)
            current_word = token[1:]  # Remove ▁
            current_word_has_rationale = (z_val == 1.0)
        else:
            # Continuation of current word
            current_word += token
            if z_val == 1.0:
                current_word_has_rationale = True
    
    # Don't forget the last word
    if current_word:
        words.append(current_word)
        word_rationales.append(1 if current_word_has_rationale else 0)
    
    # Join words with spaces to form normal text
    normal_text = ' '.join(words)
    
    return normal_text, word_rationales

# Apply the processing to all rows
df['normal_text'] = ''
df['word_rationales'] = ''

for idx, row in df.iterrows():
    normal_text, word_rationales = process_text_and_rationales(row['texts'], row['z'])
    df.at[idx, 'normal_text'] = normal_text
    df.at[idx, 'word_rationales'] = word_rationales

print("Text processing completed!")
print(f"Sample processed text:\n{df.iloc[0]['normal_text']}")
print(f"Original tokens: {len(df.iloc[0]['texts'].split())}")
print(f"Words after processing: {len(df.iloc[0]['normal_text'].split())}")
print(f"Word rationales: {df.iloc[0]['word_rationales'][:10]}...")  # Show first 10

Text processing completed!
Sample processed text:
there may not be a critic alive who harbors as much affection for shlock monster movies as i do . i delighted in the sneaky - smart entertainment of ron underwood 's big - underground - worm yarn tremors ; i even giggled at last year 's critically - savaged big - underwater - snake yarn anaconda . something about these films causes me to lower my inhibitions and return to the saturday afternoons of my youth , spent in the company of ghidrah , the creature from the black lagoon and the blob . deep rising , a big - undersea - serpent yarn , does n't quite pass the test . sure enough , all the modern monster movie ingredients are in place : a conspicuously multi - ethnic / multi - national collection of bait . .. excuse me , characters ; an isolated location , here a derelict cruise ship in the south china sea ; some comic relief ; a few cgi - enhanced gross - outs ; and at least one big explosion . there are too - cheesy - to - be - accid

In [8]:
# Save the processed DataFrame with rationales
save_path = "processed_movies_rationales.csv"
df.to_csv(save_path, index=False)
print(f"Saved processed DataFrame to {save_path}")
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Saved processed DataFrame to processed_movies_rationales.csv
DataFrame shape: (199, 6)
Columns: ['texts', 'labels', 'predictions', 'z', 'normal_text', 'word_rationales']
